# Project Association: Expert Discovery System and Collaboration Network Analysis with HORUS data

# 1. Setup and imports

In [2]:
import re
import os
import unicodedata
import numpy as np
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from collections import Counter
from pathlib import Path


print("All imports successful.")

All imports successful.


### 1.1. Environment Configuration (Mounting Drive)

In [3]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    base_path = Path("/content/drive/Shareddrives/Minería/proyecto_horus/data/")
else:
    base_path = Path("../data/")

### 1.2. Coauthor cleaning functions

Defines the preprocessing logic for coauthor data. The *normalize_for_match* function standardizes names by converting them to lowercase, removing accents, punctuation, and extra whitespace; *clean_coauthors_for_rules* function applies this normalization and ensures that each publication is represented as a clean list of unique authors. This step is critical to avoid treating the same author as multiple different entities due to formatting inconsistencies.

In [4]:
# 1.3. Coauthor cleaning functions
def normalize_for_match(name):
    name = str(name).lower().strip()
    name = ''.join(c for c in unicodedata.normalize('NFD', name)
                   if unicodedata.category(c) != 'Mn')
    name = re.sub(r'[.,]', '', name)
    return ' '.join(name.split())

def clean_coauthors_for_rules(text):
    if pd.isna(text):
        return []
    
    authors = [
        normalize_for_match(name)
        for name in str(text).split(',')
        if name.strip()
    ]
    
    return list(set(authors))

In [6]:
# Load raw faculty datasets
products_arts = pd.read_excel(base_path / "raw/productos_artes.xlsx").assign(faculty='Artes')
products_agricultural_sciences = pd.read_excel(base_path / "raw/productos_ciencias_agrarias.xlsx").assign(faculty='Ciencias Agrarias')
products_economic_sciences = pd.read_excel(base_path / "raw/productos_ciencias_economicas.xlsx").assign(faculty='Ciencias Económicas')
products_human_sciences = pd.read_excel(base_path / "raw/productos_ciencias_humanas.xlsx").assign(faculty='Ciencias Humanas')
products_sciences = pd.read_excel(base_path / "raw/productos_ciencias.xlsx").assign(faculty='Ciencias')
products_law = pd.read_excel(base_path / "raw/productos_derecho.xlsx").assign(faculty='Derecho')
products_nursing = pd.read_excel(base_path / "raw/productos_enfermeria.xlsx").assign(faculty='Enfermería')
products_medicine = pd.read_excel(base_path / "raw/productos_medicina.xlsx").assign(faculty='Medicina')
products_dentistry = pd.read_excel(base_path / "raw/productos_odontologia.xlsx").assign(faculty='Odontología')
products_veterinary = pd.read_excel(base_path / "raw/productos_veterinaria.xlsx").assign(faculty='Veterinaria')

# Combine datasets
products_bogota = pd.concat([
    products_arts,
    products_agricultural_sciences,
    products_economic_sciences,
    products_human_sciences,
    products_sciences,
    products_law,
    products_nursing,
    products_medicine,
    products_dentistry,
    products_veterinary
], ignore_index=True)

# Rename columns to standardized format
column_mapping = {
    'Título original': 'original_title',
    'Descripción original': 'original_description',
    'Revista / Conferencia': 'journal_conference',
    'Coautores': 'coauthors',
    'Doi': 'doi',
    'ISBN': 'isbn',
    'ISSN': 'issn',
    'Citaciones': 'citations',
    'Idioma': 'language',
    'Fecha': 'date',
    'Tipo': 'type',
    'Fuente': 'source',
    'Enlace': 'link',
    'faculty': 'faculty'
}

products_bogota = products_bogota.rename(columns=column_mapping)

research_professors = pd.read_csv(base_path / 'external/docentes_investigadores.csv')
column_mapping_2 = {
    'Nombre': 'name',
    'Vinculación': 'affiliation',
    'Cantidad de productos': 'product_count'
}
research_professors.rename(columns=column_mapping_2, inplace=True)
active_research_professors = research_professors[research_professors['affiliation'] == 'Activo'].reset_index(drop=True)

print(f"Dataset created from raw files: {active_research_professors.shape}")
active_research_professors.head()

Dataset created from raw files: (1929, 3)


,name,affiliation,product_count
0,Milanes Carreño Diego Alejandro,Activo,979
1,Sandoval Usme Carlos Eduardo,Activo,942
2,Giraldo Gutierrez Liliana,Activo,558
3,Martinez Rodriguez Fleming,Activo,514
4,Sánchez Pedraza Ricardo,Activo,498


# 2. Create transactions

transactions = products_bogota['coauthors'].apply(clean_coauthors_for_rules)

print("Sample transactions:")
transactions.head()

In [8]:
transactions = active_research_professors.apply(clean_coauthors_for_rules)

print("Sample transactions:")
transactions.head()

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

### 2.1. Filter low-frequency authors

This step removes authors with very low frequency across the dataset. Rare authors contribute little to meaningful pattern discovery while significantly increasing computational complexity. By filtering them out, the dataset becomes more manageable, improving both performance and the quality of the extracted rules.

In [ ]:
# 3. Filter low-frequency authors
all_authors = [a for trans in transactions for a in trans]
author_counts = Counter(all_authors)
# 2. Create filtered transactions (ACTIVE COAUTHORS ONLY + FREQUENCY REDUCTION)

# --- Step 1: Build active professors set ---
active_professors_names = set(
    active_research_professors['name']
    .dropna()
    .apply(normalize_for_match)
)

# --- Step 2: Extract and clean coauthors ---
transactions = products_bogota['coauthors'].apply(clean_coauthors_for_rules)

# --- Step 3: Keep only active coauthors ---
transactions_active = transactions.apply(
    lambda authors: [a for a in authors if a in active_professors_names]
)

# --- Step 4: Remove empty transactions (no active authors) ---
transactions_active = transactions_active[transactions_active.apply(len) > 0]

print(f"Transactions with active authors: {len(transactions_active)}")

# --- Step 5: Frequency filtering (CRITICAL for Apriori performance) ---
all_authors = [author for trans in transactions_active for author in trans]

author_counts = Counter(all_authors)

# Threshold: adjust depending on dataset size
MIN_FREQ = 5  # <- puedes subirlo a 10 si sigue lento

frequent_authors = {
    author for author, count in author_counts.items()
    if count >= MIN_FREQ
}

print(f"Frequent authors retained: {len(frequent_authors)}")

# --- Step 6: Filter transactions again with frequent authors ---
transactions_filtered = transactions_active.apply(
    lambda authors: [a for a in authors if a in frequent_authors]
)

# Remove empty again
transactions_filtered = transactions_filtered[
    transactions_filtered.apply(len) > 1  # mínimo 2 para reglas
]

print(f"Final transactions after filtering: {len(transactions_filtered)}")

transactions_filtered.head()

print(f"Unique authors before: {len(set(all_authors))}")
print(f"Unique authors after: {len(valid_authors)}")

Unique authors before: 104074
Unique authors after: 18666


### 2.2. Transaction Encoding

This step converts the transactional data into a one-hot encoded matrix. Each column represents an author, and each row indicates whether that author appears in a given publication. This binary format is required by association rule algorithms such as Apriori and FP-Growth.

In [ ]:
# 4. Transaction Encoding
te = TransactionEncoder()
te_array = te.fit(transactions_filtered).transform(transactions_filtered)

df_encoded = pd.DataFrame(te_array, columns=te.columns_)

print(df_encoded.shape)
df_encoded.head()

(134598, 18666)


,a,a a,a a ramirez,a abidin,a acuna,a ahmed,a b,a bermudez,a bustinza,a c,...,zuniga rodriguez eduardo adolfo,zur nedden m,zurita vanegas jorge humberto,zurnedden m,zurzolo g,zutshi v,zverev e,zvyagin a,zwalinski,zwalinski l
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


: 

# 3. Apriori

The Apriori algorithm is applied to identify frequent combinations of authors. It works by iteratively generating candidate itemsets and evaluating their support. While simple and interpretable, Apriori can be computationally expensive for large datasets.

In [ ]:
# 5. Apriori
frequent_itemsets_apriori = apriori(
    df_encoded,
    min_support=0.001,
    use_colnames=True
)

print(frequent_itemsets_apriori.head())

# 4. FP-Growth

FP-Growth is applied as a more efficient alternative to Apriori. It uses a compressed tree structure to identify frequent patterns without generating all candidate combinations. This results in significantly faster execution while producing equivalent results.

In [ ]:
# 6. FP-Growth
frequent_itemsets_fp = fpgrowth(
    df_encoded,
    min_support=0.001,
    use_colnames=True
)

print(frequent_itemsets_fp.head())

# 5. Generate rules

This step generates association rules from the frequent itemsets. Each rule represents a relationship between sets of authors. The rules are filtered using the lift metric, which measures how much stronger the association is compared to random chance.

In [ ]:
# 7. Generate association rules
rules = association_rules(
    frequent_itemsets_fp,
    metric="lift",
    min_threshold=1.2
)

print(rules.head())

### 5.1. Top rules

The rules are ranked based on their importance using lift, confidence, and support. Lift is prioritized as it reflects the strength of the association beyond random chance. The top 10 rules represent the most significant collaboration patterns in the dataset.

In [ ]:
# 8. Sort and select top rules
rules_sorted = rules.sort_values(
    by=["lift", "confidence", "support"],
    ascending=False
)

top_10 = rules_sorted.head(10)

top_10[['antecedents', 'consequents', 'support', 'confidence', 'lift']]